# Wizard RL — ExperimentsLaunch the ablation and rebuild the figures from its results.The five configurations are additive: each adds exactly one axis to the onebefore it, so a difference in the outcome is attributable to that axis.| | bid_mode | value baseline | round sampling | tricks aux ||---|---|---|---|---|| **A** | policy | off | uniform | off || **B** | policy | **on** | uniform | off || **C** | policy | on | **r²** | off || **D** | policy | on | r² | **on** || **E** | **ev** | on | r² | on || **F** | ev | on | r² | on | *(plus heuristic training opponents)* |`p_heur = 0` for A–E: the story predates heuristic opponents, and mixing themin would add a variable that is not part of it. F is that variable on its own.**Runs are launched as subprocesses, not in notebook cells.** A run takeshours; a dying kernel must not take it with it.

## Setup

In [ ]:
import json, os, subprocess, sys, timefrom pathlib import Pathimport numpy as npimport matplotlib.pyplot as pltREPO = Path.cwd()PYTHON = str(REPO / ".wizard_venv" / "bin" / "python")RESULTS = REPO / "results"LOGS = REPO / "run_logs"; LOGS.mkdir(exist_ok=True)sys.path.insert(0, str(REPO))from train import EXPERIMENTS, Config, load_resultsfor name, cfg in EXPERIMENTS.items():    if name != "default":        print(f"{name}: {cfg.run_suffix}")

## 1 — LaunchEach cell call starts one detached run. Stdout goes to `run_logs/<name>.log`,results to `results/<name>.json`, TensorBoard to the usual place.Start them one at a time if the machine is busy, or all at once if it is not —they are independent.

In [ ]:
def launch(name, updates=None, seed=None):    """Start one configuration detached. Returns the Popen handle."""    cmd = [PYTHON, "train.py", "--config", name]    if updates is not None: cmd += ["--updates", str(updates)]    if seed is not None:    cmd += ["--seed", str(seed)]    log = open(LOGS / f"{name}.log", "w")    p = subprocess.Popen(cmd, cwd=REPO, stdout=log, stderr=subprocess.STDOUT,                         start_new_session=True)    print(f"{name} started, pid {p.pid}, log -> run_logs/{name}.log")    return p# Smoke first: two updates each, confirms every configuration runs at all.# procs = {n: launch(n, updates=2) for n in "ABCDE"}

In [ ]:
# The real thing. Uncomment when the smoke run came back clean.# procs = {n: launch(n) for n in "ABCDE"}

## 2 — Status

In [ ]:
def status():    rows = []    for name in "ABCDEF":        res = load_results(name, result_dir=str(RESULTS))        if res is None:            rows.append((name, "-", "-", "-")); continue        h = res["history"]        last = h[-1] if h else {}        rows.append((name, len(h), last.get("update", "-"),                     f"{last.get('score_vs_random', float('nan')):.1f}"))    print(f"{'cfg':>4} {'evals':>6} {'update':>7} {'vs random':>10}")    for r in rows:        print(f"{r[0]:>4} {str(r[1]):>6} {str(r[2]):>7} {str(r[3]):>10}")status()

## 3 — FiguresOne figure per row of the results table, each carrying exactly one claim.Everything is rebuilt from `results/*.json`, so the figures are reproducibleand no screenshot is involved.

In [ ]:
def series(name, key):    res = load_results(name, result_dir=str(RESULTS))    if not res or not res["history"]:        return np.array([]), np.array([])    h = res["history"]    xs = np.array([r["update"] for r in h if key in r])    ys = np.array([r[key] for r in h if key in r], dtype=float)    return xs, ysdef compare(key, title, ylabel, configs="ABCDE", hline=None, hlabel=None, ax=None):    ax = ax or plt.subplots(figsize=(7, 4))[1]    for name in configs:        xs, ys = series(name, key)        if len(xs):            ax.plot(xs, ys, label=name, linewidth=1.6)    if hline is not None:        ax.axhline(hline, ls="--", c="0.4", lw=1, label=hlabel)    ax.set_title(title); ax.set_xlabel("update"); ax.set_ylabel(ylabel)    ax.legend(); ax.grid(alpha=.3)    return ax

In [ ]:
# Headline: the axis that is never trained against.compare("score_vs_random", "Score against random opponents", "points / game",        hline=223.3, hlabel="heuristic reference (223.3)")plt.tight_layout(); plt.show()

In [ ]:
# A / C / E: the bidding ceiling in round 20. This is the central pathology,# and it now comes from the bids actually played, so it is comparable across# the policy configurations and the EV one.fig, axes = plt.subplots(1, 2, figsize=(13, 4))compare("bids_max_r20", "Highest bid in round 20", "bid", ax=axes[0])compare("bids_mean_r20", "Mean bid in round 20", "bid", ax=axes[1],        hline=6.67, hlabel="structural expectation 6.67")plt.tight_layout(); plt.show()

In [ ]:
# B and C: does the collapse move? The rank ratio is the share of the bid# logits' variation living in a single direction -- it rises as the head stops# distinguishing hands. Only defined where the bid head is a policy.fig, axes = plt.subplots(1, 2, figsize=(13, 4))compare("rank_bid_all", "Rank ratio of the bid logits (collapse)", "s0 / sum(s)",        configs="ABCD", ax=axes[0])compare("bids_std_r20", "Spread of the round-20 bids", "std", ax=axes[1],        hline=1.50, hlabel="spread of E[W|hand] (1.50)")plt.tight_layout(); plt.show()

In [ ]:
# D: how good is the tricks head at bid time? The bid in configuration E# depends entirely on it.compare("tricks_mae_r20_at_bid", "Trick prediction error at bid time, round 20",        "MAE (tricks)", configs="DE")plt.tight_layout(); plt.show()

In [ ]:
# E: the bid distribution over training. At update 0 the tricks head is# uninformative, q is uniform, and the EV rule has a fixed point near 10 --# that spike is a property of the scoring function, not a learned bid.res = load_results("E", result_dir=str(RESULTS))if res and res["bids_r20"]:    steps = sorted(res["bids_r20"], key=int)    show = [steps[0]] + steps[len(steps)//2::max(1, len(steps)//3)]    fig, ax = plt.subplots(figsize=(8, 4))    for s in dict.fromkeys(show):        ax.hist(res["bids_r20"][s], bins=np.arange(-0.5, 21.5), histtype="step",                density=True, linewidth=1.6, label=f"update {s}")    ax.axvline(6.67, ls="--", c="0.4", lw=1, label="expectation 6.67")    ax.set_title("Round-20 bid distribution over training")    ax.set_xlabel("bid"); ax.set_ylabel("density"); ax.legend(); ax.grid(alpha=.3)    plt.tight_layout(); plt.show()else:    print("no histograms yet -- they are written at every 250th update")

## 4 — Table for the READMEThe numbers behind the Experiments section, taken from the last eval point ofeach run.

In [ ]:
KEYS = [("score_vs_random", "vs random"), ("score_vs_heuristic", "vs heuristic"),        ("bids_max_r20", "max bid r20"), ("bids_mean_r20", "mean bid r20"),        ("bids_std_r20", "std bid r20"), ("acc_r20", "hit rate r20"),        ("tricks_mae_r20_at_bid", "tricks MAE r20")]hdr = f"{'cfg':<4}" + "".join(f"{lbl:>16}" for _, lbl in KEYS)print(hdr); print("-" * len(hdr))for name in "ABCDEF":    res = load_results(name, result_dir=str(RESULTS))    if not res or not res["history"]:        continue    last = res["history"][-1]    row = f"{name:<4}"    for key, _ in KEYS:        v = last.get(key)        row += f"{v:>16.2f}" if isinstance(v, (int, float)) else f"{'-':>16}"    print(row)